### Ingest Winning Bidders CSV
This notebook ingests the winning bidders file from the landing layer, standardizes the dataset, and writes it to a bronze Delta table.

Business context:
* This table shows the bidders who won lots during the auction
* Each row links a sold lot to the winning bidder and the related sale outcome
* The bronze table preserves these raw results so they can support downstream reporting, fulfillment, and bidder analysis

#### Step 1: Load shared setup values
This step imports the common project configuration used across bronze ingestion notebooks.

What happens here:
* Loads shared Unity Catalog and volume variables
* Reuses helper functions such as metadata column enrichment
* Keeps environment-specific values centralized in one setup notebook


#### Step 2: Build the winning bidders file path
This step assembles the full storage path for the raw winning bidders CSV.

What happens here:
* Combines the configured volume, catalog, schema, and folder names
* Points to the specific file containing auction winners
* Makes the resolved source path easy to validate before reading data


#### Step 3: Define the winning bidders schema
This step declares the expected structure of the CSV so Spark reads bidder, lot, and address fields with the intended data types.

What happens here:
* Defines the incoming winning bidder column names explicitly
* Applies consistent types for lot numbers, bidder identifiers, timestamps, and price fields
* Avoids schema drift that can happen with CSV inference alone


#### Winning bidders data dictionary
This file represents the auction results at the winning bidder level, where each record shows who won a lot and the associated commercial details.

Key fields:
* `Auction Name`: the auction event where the lot was sold
* `Auction Date`: the date of the auction event
* `Lot #`: the lot number won by the bidder
* `Item name`: the description of the lot or item sold
* `Hammer Price`: the final bid accepted at the close of bidding
* `Buyers Premium`: the buyer surcharge added on top of the hammer price
* `Price Realized`: the total realized amount for the lot
* `Status`: the sale result status for the lot
* `Online/floor`: shows whether the winning bidder participated online or on the floor
* `Bidder number` and `Registration number`: bidder identifiers used by the auction platform
* `First Name`, `Last Name`, and `Email`: key contact details for the winning bidder
* Billing and shipping fields: customer fulfillment and invoicing details for the winning lot

Business use:
* Identify which bidders won specific lots during the auction
* Support post-sale reporting, invoicing, and shipping workflows
* Analyze bidder outcomes, sales values, and channel participation across auctions


#### Step 4: Load and enrich the winning bidders data
This step reads the raw CSV into Spark and appends standard metadata columns used for traceability.

What happens here:
* Loads the winning bidders file into a Spark DataFrame
* Applies the predefined schema during ingestion
* Adds metadata such as source path and ingestion timestamp


#### Step 5: Prepare the bronze-ready winners dataset
This step standardizes the DataFrame so the columns are easier to use after the bronze write.

What happens here:
* Renames source columns into a lowercase, underscore-based format
* Cleans special characters in field names for downstream usability
* Confirms the bronze target context before persisting the table


#### Step 6: Write the bronze winning bidders table
This final step persists the cleaned winning bidders DataFrame as a Delta table in the bronze schema.

What happens here:
* Writes the transformed winning bidder data to a managed Delta table
* Rebuilds the target table on reruns using overwrite mode
* Makes the bronze dataset available for downstream silver transformations, fulfillment, and reporting


In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

In [ ]:
dbutils.widgets.text('id'," ")
batch_id = dbutils.widgets.get('id')
batch_id

In [ ]:
# Build the full source path for the CSV file to ingest from cloud storage.
# This keeps the input file location parameterized through shared environment values.
path = f'/{volume}/{catalog}/{schema}/{file}/{batch_id}/winning*.csv'

In [ ]:
# Import explicit Spark SQL data types so the CSV schema can be declared manually.
from pyspark.sql.types import *

# Define the expected structure of the winning bidders CSV.
# Using an explicit schema prevents unwanted type inference issues.
schema_winning_bidders = StructType([
    StructField('Platform', StringType(), True),
    StructField('Auction Name', StringType(), True),
    StructField('Auction Date', StringType(), True),
    StructField('Lot #', IntegerType(), True),
    StructField('Item name', StringType(), True),
    StructField('Currency', StringType(), True),
    StructField('Hammer Price', DoubleType(), True),
    StructField('Buyers Premium', DoubleType(), True),
    StructField('Price Realized', DoubleType(), True),
    StructField('Status', StringType(), True),
    StructField('Online/floor', StringType(), True),
    StructField('Bidder number', IntegerType(), True),
    StructField('Registration number', IntegerType(), True),
    StructField('First Name', StringType(), True),
    StructField('Last Name', StringType(), True),
    StructField('Email', StringType(), True),
    StructField('DateTime', TimestampType(), True),
    StructField('Commission Amount', DoubleType(), True),
    StructField('Billing Company', StringType(), True),
    StructField('Billing First name', StringType(), True),
    StructField('Billing Last name', StringType(), True),
    StructField('Billing Phone Number', StringType(), True),
    StructField('Billing Address 1', StringType(), True),
    StructField('Billing Address 2', StringType(), True),
    StructField('Billing City', StringType(), True),
    StructField('Billing State/Province', StringType(), True),
    StructField('Billing ZIP/Postal Code', StringType(), True),
    StructField('Billing Country', StringType(), True),
    StructField('Shipping Company', StringType(), True),
    StructField('Shipping First name', StringType(), True),
    StructField('Shipping Last name', StringType(), True),
    StructField('Shipping Phone Number', StringType(), True),
    StructField('Shipping Address 1', StringType(), True),
    StructField('Shipping Address 2', StringType(), True),
    StructField('Shipping City', StringType(), True),
    StructField('Shipping State/Province', StringType(), True),
    StructField('Shipping ZIP/Postal Code', StringType(), True),
    StructField('Shipping Country', StringType(), True)
])

In [ ]:
# Read the source CSV into a Spark DataFrame using the predefined schema.
# The file is expected to contain a header row.
df_winning_bidders = spark.read.format('csv') \
    .option('header', 'true') \
         .option('multiLine', 'true') \
    .option("quote", '"') \
    .option("escape", '"') \
        .option('timestampFormat', 'yyyy-MM-dd HH:mm:ss z') \
    .schema(schema_winning_bidders) \
    .load(path)


# Add standard metadata or audit columns using the shared helper function.
df_winning_bidders = add_columns(df_winning_bidders)

In [ ]:
df_winning_bidders = df_winning_bidders.withColumnsRenamed({col: col.replace(' ', '_').lower() for col in df_winning_bidders.columns})

df_winning_bidders = df_winning_bidders.withColumnsRenamed({'online/floor':'online_floor', 'billing_state/province': 'billing_state_province', 'billing_zip/postal_code': 'billing_zip_postal_code','shipping_state/province':'shipping_state_province', 'shipping_zip/postal_code': 'shipping_zip_postal_code'})

In [ ]:
# Preview the target bronze namespace components before writing the Delta table.
f'{volume}/{catalog}/{bronze}'

In [ ]:
# Write the ingested DataFrame to the bronze layer as a Delta table.
# Overwrite mode is used so the table can be rebuilt on repeated runs.
df_winning_bidders \
    .write \
    .format('delta') \
    .mode('overwrite') \
    .partitionBy('batch_id') \
    .option('replaceWhere',f"batch_id = '{batch_id}'")\
    .saveAsTable(f'{catalog}.{bronze}.winningbidders')